# 02 — Eğitim ve değerlendirme (GPU)
**Çalışma zamanı → GPU** (T4, L4 veya A100; hangisi olduğu kurulum hücresinde yazar).
Secrets: `KAGGLE_API_TOKEN`, `GITHUB_TOKEN`.
Sıra: smoke → 1 epoch (süre ölç) → Claude'a getir → tam eğitim → değerlendirme.
Veri: private Kaggle veri seti `ouzhantarhan/baykus-dv-rgbt` (ADR 0004), birkaç dakikada iner.
Çıktılar Drive'da `MyDrive/baykus/{runs,eval,results}`. Her epoch `weights/last.pt` olarak
Drive'a yazılır; oturum koparsa kurulum hücrelerinden sonra "3b) devam" hücresi.

In [ ]:
CONFIG = 'configs/e1_rgb.yaml'   # e2_ir.yaml, e3_early_fusion.yaml ...
BRANCH = 'main'                  # haftalık dalı denemek için ör. 'h3'
DRIVE = '/content/drive/MyDrive/baykus'
from google.colab import drive
drive.mount('/content/drive')

import glob, json, os
from pathlib import Path

def runs(finished: bool) -> list[str]:
    # Bu config'in tam eğitim klasörleri (smoke ve 1 epoch hariç), eskiden yeniye
    slug = Path(CONFIG).stem.replace('_', '-')
    out = []
    for d in sorted(glob.glob(f'{DRIVE}/runs/*_{slug}_*')):
        if d.endswith(('_smoke', '_ep1')) or not os.path.exists(f'{d}/weights/last.pt'):
            continue
        rec = f'{d}/baykus_train.json'
        status = json.load(open(rec, encoding='utf-8')).get('status') if os.path.exists(rec) else None
        if (status == 'finished') == finished:
            out.append(d)
    return out

In [ ]:
import os
from google.colab import userdata
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
print('token yüklendi (değer yazdırılmadı)')

In [ ]:
# Repo private ise Secrets'a GITHUB_TOKEN ekle (fine-grained, sadece bu repo, 'Contents: read').
import os
from google.colab import userdata
try:
    tok = userdata.get('GITHUB_TOKEN')
except Exception:
    tok = None
url = f'https://{tok}@github.com/OgzhnTarhn/baykus.git' if tok else 'https://github.com/OgzhnTarhn/baykus.git'
if not os.path.isdir('/content/baykus/.git'):
    !git clone -q {url} /content/baykus
!git -C /content/baykus fetch -q {url} {BRANCH}
!git -C /content/baykus checkout -q -B {BRANCH} FETCH_HEAD
!git -C /content/baykus remote set-url origin https://github.com/OgzhnTarhn/baykus.git
%cd /content/baykus
!git log --oneline -3

In [ ]:
%%time
!pip -q install uv kaggle
!python --version   # Spike C: Colab görüntüsü zamanla değişir
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!nproc              # CPU çekirdeği: veri yükleme hızını belirler (gpu-budget.md)
!uv pip install --system -q -e '/content/baykus/python[train]'
import yaml
MODE = yaml.safe_load(open(f'/content/baykus/{CONFIG}', encoding='utf-8'))['mode']
print('MODE =', MODE)

In [ ]:
%%time
# İşlenmiş veri: private Kaggle veri seti (ADR 0004). Kaggle yüklenen tar'ları klasöre açtı;
# CLI klasör seçerek indiremediği için veri setinin tamamı iner (~9,8 GB).
DS = 'ouzhantarhan/baykus-dv-rgbt'
!kaggle datasets download {DS} -p /content/dv_rgbt --unzip -q
!mv /content/dv_rgbt/meta /content/meta
!baykus-data illum-relabel --meta /content/meta   # ADR 0005: Kaggle'daki etiketler eski eşikle
!baykus-data slices --out /content/dv_rgbt --meta /content/meta
!baykus-data yaml --out /content/dv_rgbt
!du -sh /content/dv_rgbt/* /content/meta
from pathlib import Path
NEED = {'rgb': ['rgb'], 'ir': ['ir'], 'early_fusion': ['rgb', 'ir']}[MODE]
# H1 Görev 10'dan taşındı: kutular araçların üstüne oturuyor mu? (gözle)
# Her set kendi görüntüsünde: solda RGB etiketleri RGB'de, sağda IR etiketleri IR'de.
import random, cv2, numpy as np
from IPython.display import Image, display
random.seed(1)
root = Path('/content/dv_rgbt')

def boxed(img_path, label_path):
    im = cv2.imread(str(img_path), cv2.IMREAD_COLOR)  # IR gri → 3 kanal
    h_im, w_im = im.shape[:2]
    lines = label_path.read_text(encoding='utf-8').splitlines() if label_path.exists() else []
    for ln in lines:
        c, cx, cy, w, h = map(float, ln.split())
        cv2.rectangle(im, (int((cx - w / 2) * w_im), int((cy - h / 2) * h_im)),
                      (int((cx + w / 2) * w_im), int((cy + h / 2) * h_im)), (0, 255, 0), 1)
    return im, len(lines)

for s in random.sample(sorted(p.stem for p in (root / 'rgb/images/train').glob('*.jpg')), 3):
    rgb, n_rgb = boxed(root / f'rgb/images/train/{s}.jpg', root / f'labels_rgb/train/{s}.txt')
    ir, n_ir = boxed(root / f'ir/images/train/{s}.png', root / f'labels_ir/train/{s}.txt')
    print(f'{s}: RGB {n_rgb} kutu, IR {n_ir} kutu')
    cv2.imwrite(f'/content/check_{s}.jpg', np.hstack([rgb, ir]))
    display(Image(f'/content/check_{s}.jpg'))
for d in NEED:  # beklenen düzen: <modalite>/images/{train,val,test} (Data Explorer'da görüldü)
    for split in ('train', 'val', 'test'):
        assert Path(f'/content/dv_rgbt/{d}/images/{split}').is_dir(), f"{d}/images/{split} yok; ls çıktısını Claude'a getir"

In [ ]:
%%time
if MODE == 'early_fusion':  # rgbt4 yüklenmedi (ADR 0004); H3'te bu hücre yeniden ele alınır
    !baykus-data rgbt4 --out /content/dv_rgbt

In [ ]:
%%time
# 1) smoke: %5 veri, 1 epoch — hattın uçtan uca çalıştığını doğrular
!cd /content/baykus/python && baykus-train --config ../{CONFIG} --data-root /content/dv_rgbt --project {DRIVE}/runs --results {DRIVE}/results --smoke

In [ ]:
%%time
# 2) 1 tam epoch — süresi GPU bütçesine yazılır (spec §10.2)
!cd /content/baykus/python && baykus-train --config ../{CONFIG} --data-root /content/dv_rgbt --project {DRIVE}/runs --results {DRIVE}/results --epochs 1

🚧 **KAPI: Burada dur.** Yukarıdaki iki hücrenin süresini Claude'a getir; bütçe onaylanınca devam.

In [ ]:
# 3) tam eğitim (config'teki epoch sayısı)
!cd /content/baykus/python && baykus-train --config ../{CONFIG} --data-root /content/dv_rgbt --project {DRIVE}/runs --results {DRIVE}/results

In [ ]:
# 3b) devam: oturum koptuysa önce Drive → token → klonlama → kurulum → veri hücrelerini
# çalıştır, sonra bunu. Bu config'in yarım kalan en son eğitimi kaldığı epoch'tan sürer.
todo = runs(finished=False)
print('devam edilecek:', todo[-1] if todo else 'yok')
if todo:
    RUN_DIR = todo[-1]
    !cd /content/baykus/python && baykus-train --config ../{CONFIG} --resume "{RUN_DIR}" --data-root /content/dv_rgbt --results {DRIVE}/results

In [ ]:
# 4) değerlendirme (val): bu config'in biten en son tam eğitiminin best.pt'si
done = runs(finished=True)
assert done, 'biten tam eğitim yok; önce 3) veya 3b) bitmeli'
W = f'{done[-1]}/weights/best.pt'
print(W)
!cd /content/baykus/python && baykus-eval run --config ../{CONFIG} --weights "{W}" --data-root /content/dv_rgbt --meta /content/meta --project {DRIVE}/eval --results {DRIVE}/results